In [9]:
from modelling import obj_to_volume, modelling_script
from weight_calculation import weight_formula
from segmentation import segmentation_script
import yaml
import trimesh
import pymeshfix
import os   
import time
import importlib
import pandas as pd
import numpy as np

CONFIG_PATH = "/Users/adeleyounis/Desktop/Capstone/wAI/config.yaml"

def load_config(config_path: str):
    """Load and parse YAML configuration."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    base_dir = os.getcwd()
    def resolve_path(rel_path):
        if os.path.isabs(rel_path):
            return rel_path
        return os.path.normpath(os.path.join(base_dir, rel_path))

    # Resolve all paths
    paths = {k: resolve_path(v) for k, v in config.get("paths", {}).items()}
    return paths, config

In [ ]:
def main(
    rgb_path: str = None,
    depth_path: str = None,
    sex: str = None,
    height: float = None,
    file_name: str = None,
    visualize: bool = False,
    save: bool = True,
):
    """
    Run full wAI software pipeline with specified paths and inputs.
    If paths are not provided, falls back to config.yaml.
    """
    paths, config = load_config(CONFIG_PATH)
    
    # Use provided paths or fall back to config
    rgb_img_path = rgb_path or paths["rgb_img_path"]
    depth_img_path = depth_path or paths["depth_img_path"]
    
    # Use provided inputs or fall back to config
    sex_value = sex or config["inputs"]["sex"]
    height_value = height or config["inputs"]["height"]
    
    # 1) Segmentation pipeline (0-50%)
    point_cloud, img_rgb, person_segmentation_mask, x1, y1, x2, y2 = segmentation_script.run_pipeline(
        frame_rgb=rgb_img_path, 
        depth_arr=depth_img_path,
        config=config,
        file_name=file_name,
        visualize=visualize, 
        save=save
        )
    
    # 2) Modelling pipeline (50-85%)
    mesh = modelling_script.main(
        img_rgb=img_rgb, x1=x1, y1=y1, x2=x2, 
        y2=y2, point_cloud=point_cloud, person_segmentation_mask=person_segmentation_mask,
        visualize=visualize, save=save)

    # 3) Volume calculation (85-90%)
    vol = mesh.get_volume() * 1000
    print(f"Volume: {vol} cm³")

    # 4) Weight estimation (90-100%)
    print("Weight using Open3D volume:")
    weight_result = weight_formula.able_body_weight_formula(
        sex=sex_value, 
        volume=vol, 
        height=height_value)
    
    return {
        "volume": vol,
        "weight": weight_result,
        "sex": sex_value,
        "height": height_value
    }

In [10]:
data_csv = "/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/data/captures.csv"
base_dir = "/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/"

df = pd.read_csv(data_csv)

def parse_height(height_str):
    try:
        return float(str(height_str).replace("cm", "").strip())
    except:
        return None
    
def extract_uid(rgb_path):
    filename = os.path.basename(rgb_path)
    return filename.replace("_rgb.png", "")

df["height_cm"] = df["height"].apply(parse_height)

In [ ]:
# Batch Process data to save meshes and point clouds
def run_model(id, rgb_path, depth_path, sex, height_cm):
    print(f"Processing {id}")
    main(rgb_path, depth_path, sex, height_cm, file_name=id)

for _, row in df.iterrows():
    rgb_rel = row["rgb_path"]
    depth_rel = row["depth_path"]

    rgb_full = os.path.join(base_dir, rgb_rel.replace("\\", os.sep))
    depth_full = os.path.join(base_dir, depth_rel.replace("\\", os.sep))
    print(rgb_full)
    if os.path.exists(rgb_full) and os.path.exists(depth_full):

        uid = extract_uid(rgb_full)
        sex = row["sex"]
        height_cm = row["height_cm"]

        if pd.notna(height_cm) and pd.notna(sex):
            run_model(uid, rgb_full, depth_full, sex, height_cm)
        else:
            print(f"Skipping (missing metadata): {uid}")

In [18]:
# convert estimated_weight_kg to estimated_weight_ibs
df["estimated_weight_lbs"] = df["estimated_weight_kg"] * 2.20462

# get accuracy between weight and estimated_weight_ibs
def compute_metrics(true, pred):
    mae = np.mean(np.abs(true - pred))
    rmse = np.sqrt(np.mean((true - pred)**2))
    accuracy_rate = np.abs(true - pred) <= (0.10 * true)
    accuracy = np.mean(accuracy_rate) * 100
    return mae, rmse, accuracy

def compute_error_stats(true, pred):
    errors = pred - true
    std_error = np.std(errors, ddof=1)   # sample standard deviation
    mean_error = np.mean(errors)         # bias
    return mean_error, std_error

# compare across the three bulky_level [0,1,2]
results = {}
grouped = df.groupby(["bulky_level", "sex"])

for (level, sex), subset in grouped:
    
    true = subset["weight"].values
    pred = subset["estimated_weight_lbs"].values
    
    mae, rmse, accuracy = compute_metrics(true, pred)
    mean_err, std_err = compute_error_stats(true, pred)
    
    results[(level, sex)] = {
        "MAE": mae,
        "RMSE": rmse,
        "Accuracy (PW10)": accuracy,
        "Mean Error (Bias)": mean_err,
        "Standard deviation": std_err,
        "N Samples": len(subset)
    }

results_df = pd.DataFrame(results).T
print(results_df)

                MAE       RMSE  Accuracy (PW10)  Mean Error (Bias)  \
0 female  23.476561  26.645360        37.500000          17.942459   
  male    10.769849  15.480218        84.615385          -2.156752   
1 female  20.535883  23.345231        42.857143          20.535883   
  male    19.884079  25.886843        66.666667          16.883077   
2 female  14.696533  18.823970        50.000000          14.696533   
  male    14.046170  15.864033        75.000000          10.227868   

          Standard deviation  N Samples  
0 female           21.058920        8.0  
  male             15.955178       13.0  
1 female           11.992644        7.0  
  male             20.496306       12.0  
2 female           13.582036        4.0  
  male             14.002774        4.0  
